# 04 — Honest Evaluation

**Goal:** Rigorously evaluate anomaly detection models with proper methodology.

### Why Most Anomaly Detection Evaluations Are Misleading

Common mistakes in tutorials and papers:
1. **Training on the same data you test on** — inflates all metrics
2. **Labeling entire date ranges as anomalous** — not every day in a "crash" is equally anomalous
3. **Setting contamination = known anomaly rate** — leaks ground truth to the model
4. **No baseline comparison** — impossible to tell if ML adds value over simple rules
5. **Reporting only AUC** — a metric that hides poor precision in imbalanced datasets

### Our Methodology
- **Temporal train/test split** — train on data before July 2024, test on after
- **Point-level labels** — a day is anomalous only if |return| > 3% OR volume z-score > 3
- **Naive baseline** — simple threshold rule as the bar to beat
- **Multiple metrics** — precision, recall, F1, AUC, and detection latency
- **Honest reporting** — we show the real (often ugly) numbers

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import DataFetcher
from src.data.feature_engineer import FeatureEngineer
from src.models.naive_baseline import NaiveBaselineDetector
from src.models.statistical import StatisticalDetector
from src.models.isolation_forest import IsolationForestDetector
from src.models.ensemble import EnsembleDetector
from src.evaluation.metrics import AnomalyEvaluator
from src.utils.helpers import load_config

plt.style.use('seaborn-v0_8-darkgrid')
config = load_config('../config/config.yaml')
eval_cfg = config['evaluation']
evaluator = AnomalyEvaluator()

## 1. Data Preparation with Temporal Split

In [ ]:
ticker = 'SPY'
fetcher = DataFetcher(config)
df = fetcher.fetch(ticker=ticker, period='5y')
fe = FeatureEngineer(config)
featured = fe.engineer(df)
feature_cols = [c for c in fe.get_feature_columns() if c in featured.columns]

# Temporal split
train_end = eval_cfg.get('train_end', '2024-06-30')
test_start = eval_cfg.get('test_start', '2024-07-01')
train_df, test_df = evaluator.temporal_split(featured, train_end, test_start)

# Point-level labels on TEST set only
point_cfg = eval_cfg.get('point_label', {})
y_true = evaluator.label_points(
    test_df,
    min_abs_return=point_cfg.get('min_abs_return', 0.03),
    min_volume_zscore=point_cfg.get('min_volume_zscore', 3.0),
)

print(f"Train set: {len(train_df)} days ({train_df.index.min().date()} to {train_df.index.max().date()})")
print(f"Test set:  {len(test_df)} days ({test_df.index.min().date()} to {test_df.index.max().date()})")
print(f"\nTest set anomaly labels:")
print(f"  Normal days:    {(y_true == 0).sum()}")
print(f"  Anomalous days: {(y_true == 1).sum()}")
print(f"  Anomaly rate:   {y_true.mean():.1%}")
print(f"\nLabeling criteria: |return| > {point_cfg.get('min_abs_return', 0.03):.0%} OR volume z-score > {point_cfg.get('min_volume_zscore', 3.0)}")

## 2. Train on Train, Predict on Test

Models see ONLY the training data. Predictions are made on unseen test data.

In [ ]:
# === Naive Baseline (no training) ===
naive = NaiveBaselineDetector(config)
naive_result = naive.detect(test_df)
naive_pred = naive_result['anomaly'].astype(int).values
naive_scores = naive_result['anomaly_score'].values

# === Statistical (no training, applied to test) ===
stat = StatisticalDetector(config)
stat_result = stat.detect(test_df['returns'])
stat_pred = stat_result['anomaly'].astype(int).values
stat_scores = stat_result['anomaly_score'].values

# === Isolation Forest (trained on TRAIN, predict on TEST) ===
iso = IsolationForestDetector(config)
iso.fit(train_df[feature_cols])  # <-- train only
iso_result = iso.detect(test_df, feature_cols=feature_cols)
iso_pred = iso_result['anomaly'].astype(int).values
iso_scores = iso_result['anomaly_score'].values

# === Ensemble (stat + IF) ===
n = min(len(stat_scores), len(iso_scores))
ens = EnsembleDetector(config)
ens_combined = ens.combine_scores({
    'statistical': stat_scores[-n:],
    'isolation_forest': iso_scores[-n:],
})
ens_pred = (ens_combined > 0.5).astype(int)

y_true_aligned = y_true[-n:]

print("Models trained/applied. Ready for evaluation.")

## 3. Results: The Honest Numbers

In [ ]:
predictions = {
    'Naive Baseline': naive_pred[-n:],
    'Statistical': stat_pred[-n:],
    'Isolation Forest': iso_pred[-n:],
    'Ensemble': ens_pred,
}
scores_dict = {
    'Naive Baseline': naive_scores[-n:],
    'Statistical': stat_scores[-n:],
    'Isolation Forest': iso_scores[-n:],
    'Ensemble': ens_combined,
}

comparison = evaluator.compare_models(y_true_aligned, predictions, scores_dict)

# Display with context
print("=" * 70)
print("ANOMALY DETECTION — TEST SET PERFORMANCE")
print(f"Test period: {test_df.index.min().date()} to {test_df.index.max().date()}")
print(f"True anomaly rate: {y_true_aligned.mean():.1%}")
print("=" * 70)
display_cols = ['precision', 'recall', 'f1', 'auc', 'n_predicted_anomalies']
comparison[display_cols].round(3)

In [ ]:
# Does any ML model beat the baseline?
baseline_f1 = comparison.loc['Naive Baseline', 'f1']
ml_models = comparison.drop('Naive Baseline')
best_ml = ml_models['f1'].idxmax()
best_ml_f1 = ml_models.loc[best_ml, 'f1']

print(f"\n--- Baseline Comparison ---")
print(f"Naive Baseline F1: {baseline_f1:.3f}")
print(f"Best ML Model:     {best_ml} (F1: {best_ml_f1:.3f})")
if best_ml_f1 > baseline_f1:
    print(f"ML WINS by +{best_ml_f1 - baseline_f1:.3f} F1")
elif best_ml_f1 == baseline_f1:
    print(f"TIE — ML offers no advantage over the simple rule")
else:
    print(f"BASELINE WINS — the simple rule outperforms ML by +{baseline_f1 - best_ml_f1:.3f} F1")
    print("This is common. Simple rules work well when anomalies are defined by the same features the rule uses.")

## 4. ROC Curves

In [ ]:
fig = go.Figure()
colors = {'Naive Baseline': '#E91E63', 'Statistical': '#4CAF50',
          'Isolation Forest': '#FF9800', 'Ensemble': '#2196F3'}

for name, s in scores_dict.items():
    roc = evaluator.compute_roc(y_true_aligned, s)
    if len(roc['fpr']) > 0:
        fig.add_trace(go.Scatter(x=roc['fpr'], y=roc['tpr'], mode='lines',
                      name=f"{name} (AUC={roc['auc']:.3f})",
                      line=dict(color=colors.get(name, 'white'), width=2)))

fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random (AUC=0.500)',
              line=dict(color='gray', dash='dash')))

fig.update_layout(
    title='ROC Curves — Test Set (Honest Evaluation)',
    template='plotly_dark', height=500, width=600,
    xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
)
fig.show()

print("Note: AUC of 0.55-0.75 is realistic for time-series anomaly detection.")
print("If you see AUC > 0.90, the evaluation methodology likely has a flaw.")

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
model_names = list(predictions.keys())

for i, name in enumerate(model_names):
    cm = evaluator.compute_confusion_matrix(y_true_aligned, predictions[name])
    im = axes[i].imshow(cm['matrix'], cmap='Blues', aspect='auto')
    axes[i].set_title(name, fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
    axes[i].set_xticks([0, 1])
    axes[i].set_yticks([0, 1])
    axes[i].set_xticklabels(['Normal', 'Anomaly'])
    axes[i].set_yticklabels(['Normal', 'Anomaly'])
    
    # Annotate cells
    for row in range(2):
        for col in range(2):
            axes[i].text(col, row, str(cm['matrix'][row, col]),
                        ha='center', va='center', fontsize=14, fontweight='bold',
                        color='white' if cm['matrix'][row, col] > cm['matrix'].max()/2 else 'black')

plt.suptitle('Confusion Matrices — Test Set', fontsize=13)
plt.tight_layout()
plt.show()

print("Read these as: rows = actual, columns = predicted")
print("Top-left (TN): correctly identified normal days")
print("Top-right (FP): false alarms — flagged normal days as anomalous")
print("Bottom-left (FN): missed anomalies")
print("Bottom-right (TP): correctly caught anomalies")

## 6. Cross-Ticker Generalization

Does the model generalize across assets, or is it overfitting to SPY's specific patterns?

In [ ]:
# Test the SPY-trained Isolation Forest on other tickers
cross_results = []

for test_ticker in ['AAPL', 'MSFT', 'BTC-USD']:
    t_df = fetcher.fetch(ticker=test_ticker, period='5y')
    t_featured = fe.engineer(t_df)
    _, t_test = evaluator.temporal_split(t_featured, train_end, test_start)
    
    if len(t_test) < 10:
        continue
    
    t_y_true = evaluator.label_points(t_test,
                                       min_abs_return=point_cfg.get('min_abs_return', 0.03),
                                       min_volume_zscore=point_cfg.get('min_volume_zscore', 3.0))
    
    # Use the SPY-trained Isolation Forest
    t_iso_result = iso.detect(t_test, feature_cols=feature_cols)
    t_iso_pred = t_iso_result['anomaly'].astype(int).values
    t_n = min(len(t_iso_pred), len(t_y_true))
    
    metrics = evaluator.compute_metrics(t_y_true[-t_n:], t_iso_pred[-t_n:])
    roc = evaluator.compute_roc(t_y_true[-t_n:], t_iso_result['anomaly_score'].values[-t_n:])
    
    cross_results.append({
        'Ticker': test_ticker,
        'Trained On': 'SPY',
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1'],
        'AUC': roc['auc'],
        'True Anomalies': metrics['n_true_anomalies'],
        'Predicted': metrics['n_predicted_anomalies'],
    })

cross_df = pd.DataFrame(cross_results).set_index('Ticker')
print("Isolation Forest (trained on SPY) — Cross-Ticker Test Performance:\n")
cross_df.round(3)

## 7. Sensitivity Analysis

How do results change as we vary the labeling threshold? This shows how fragile or robust our conclusions are.

In [ ]:
# Vary the return threshold and see how F1 changes
thresholds = [0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05]
sensitivity_results = {name: [] for name in predictions.keys()}

for thresh in thresholds:
    y_t = evaluator.label_points(test_df, min_abs_return=thresh, min_volume_zscore=3.0)
    y_t_aligned = y_t[-n:]
    
    for name, pred in predictions.items():
        m = evaluator.compute_metrics(y_t_aligned, pred)
        sensitivity_results[name].append(m['f1'])

fig, ax = plt.subplots(figsize=(10, 5))
for name, f1s in sensitivity_results.items():
    ax.plot(thresholds, f1s, marker='o', label=name, linewidth=2)

ax.set_xlabel('Return Threshold for Anomaly Label')
ax.set_ylabel('F1 Score')
ax.set_title('Sensitivity Analysis: F1 vs Labeling Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("If model rankings change significantly with the threshold, our conclusions are fragile.")
print("If rankings are stable, the relative model performance is robust to labeling choices.")

## 8. Conclusions

### What We Found
- Anomaly detection in financial time-series is inherently hard. There is no objective ground truth.
- Our evaluation methodology (temporal split, point-level labels, baseline comparison) produces realistic metrics that reflect real-world performance.
- Model performance is sensitive to how we define "anomaly" — the sensitivity analysis shows this.

### Limitations
1. **Labels are a proxy, not truth** — defining anomalies by |return| > 3% is circular when models use returns as a feature. The naive baseline exploits this directly.
2. **Test period matters** — a calm test period has few anomalies, making metrics noisy. A volatile test period may look better but isn't generalizable.
3. **No LSTM Autoencoder in this evaluation** — the deep learning model requires separate training and may capture temporal patterns the other models miss.
4. **Single asset class** — financial markets have specific properties (fat tails, volatility clustering) that may not transfer to other domains.

### What Would Make This Better
- **Human-labeled ground truth** — have domain experts label actual anomalous days
- **Multiple test windows** — evaluate across different market regimes (bull, bear, sideways)
- **Bootstrapped confidence intervals** — show uncertainty in the metrics, not just point estimates
- **Cost-sensitive evaluation** — in practice, a missed crash costs more than a false alarm

### References
- Liu et al. (2008) — *Isolation Forest*
- Malhotra et al. (2016) — *LSTM-based Encoder-Decoder for Multi-Sensor Anomaly Detection*
- [Open Challenges in Time Series Anomaly Detection (2025)](https://arxiv.org/html/2502.05392v1) — industry perspective on evaluation problems